# 4조 · 암 아형(SUBCLASS) 분류 · 최종 제출 코드

> **이 파일 하나로 학습부터 추론까지 실행돼요.** 통합 담당: ④ 박유정 · 최종 확인: ① 장성은
> 각 담당자는 자기 칸(【② 피처】【③ 모델】【④ 경량·앙상블】)에만 최종 코드를 넣어요. 칸 밖은 고치지 않아요.

## 제출 정보 (11/11 제출 전에 채우기)
| 항목 | 내용 |
|---|---|
| 팀 | 4조 (장성은, 홍시윤, 안주형, 박유정) |
| 개발 환경(OS) | Google Colab (Linux, Python 3.13.18), CPU |
| 라이브러리 버전 | numpy 2.4.6, pandas 2.2.3, scikit-learn 1.6.1, xgboost 3.4.1, lightgbm 4.6.0, scipy 1.16.3, tqdm 4.67.3 |
| Random Seed | 42 (`SEED`) |
| 하이퍼파라미터 | 【③ 모델】 칸에 숫자로 기재 (XGB_PARAMS, LR_PARAMS, LGBM_PARAMS) |
| 모델 파일 다운로드 링크 | 없음 (노트북 안에서 학습 후 바로 예측) |
| 사전 학습 모델 | 사용 안 함 (사용 시 논문 링크와 출처 기재) |
| 외부 데이터 | 사용 안 함 (핫스팟 위치 목록은 논문 공개 지식, 아래 출처) |
| 핫스팟 출처 | Vogelstein et al., Science 2013 / Chang et al., Nat Biotechnol 2016 (cancerhotspots.org) / Yan et al., NEJM 2009 (IDH1·IDH2) / Falini et al., NEJM 2005 (NPM1) / Lee et al., PLoS Med 2006 (EGFR A289·G598) |

## 데이터 누수 방지
- 모든 전처리·피처 선택·모델 학습은 train에만 fit 해요. test에는 transform과 predict만 해요.
- test의 통계, 분포, 행 개수, index는 쓰지 않아요.

## 1. 설정

In [ ]:
import os, random, platform
import numpy as np
import pandas as pd

DATA_DIR = os.environ.get("DATA_DIR", "/data")   # 제출 규정: 데이터 입출력 경로 /data
DATA_FILES = ["train.csv", "test.csv", "sample_submission.csv"]

# 코랩에서 /data에 파일이 없을 때만 사용 (순서: 코랩 폴더 → 드라이브 → zip 업로드)
COLAB_DIRS = ["/content/data", "/content"]
DRIVE_DATA_DIR = "/content/drive/MyDrive/dacon_data"


def has_data(path):
    return all(os.path.isfile(os.path.join(path, f)) for f in DATA_FILES)


def prepare_data_dir(path):
    """기본은 path에서 읽기. 파일이 없고 코랩일 때만 코랩 폴더·드라이브·zip 업로드 순서로 찾기."""
    if has_data(path):
        return path
    try:
        import google.colab  # noqa: F401  코랩에서만 있는 모듈
    except ImportError:
        raise FileNotFoundError(f"{path}에 {DATA_FILES}가 없어요. DATA_DIR을 확인하세요.")

    # 1) 이미 코랩에 올려 둔 파일
    for d in COLAB_DIRS:
        if has_data(d):
            return d

    # 2) 구글 드라이브 (연결 거부·폴더 없음이면 다음 단계로)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        if has_data(DRIVE_DATA_DIR):
            return DRIVE_DATA_DIR
        print(f"드라이브에 {DRIVE_DATA_DIR} 데이터가 없어요 → zip 업로드로 진행")
    except Exception as e:
        print("드라이브 연결 안 함 → zip 업로드로 진행:", e)

    # 3) zip 업로드 (파일 고르는 창: 세 csv를 압축한 zip 선택)
    import zipfile
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            with zipfile.ZipFile(name) as z:
                keep = [m for m in z.namelist() if not m.startswith("__MACOSX")]   # Mac 압축 정보 제외
                z.extractall("/content/data", members=keep)
    if not has_data("/content/data"):
        raise FileNotFoundError(f"/content/data에 {DATA_FILES}가 없어요.")
    return "/content/data"


DATA_DIR = prepare_data_dir(DATA_DIR)
OUTPUT_PATH = os.path.join(DATA_DIR, "submission.csv")
print("DATA_DIR:", DATA_DIR)
SEED = 42
N_SPLITS = 5
RUN_CV = False          # True면 교차검증 점수도 다시 계산해요 (시간이 오래 걸림)
TARGET, ID_COL = "SUBCLASS", "ID"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

## 2. 환경 정보 (출력값을 위 표에 옮겨 적어요)

In [ ]:
import sklearn, xgboost, lightgbm, scipy
print("OS:", platform.platform())
print("python:", platform.python_version())
print("numpy:", np.__version__, "| pandas:", pd.__version__,
      "| scikit-learn:", sklearn.__version__, "| xgboost:", xgboost.__version__,
      "| lightgbm:", lightgbm.__version__, "| scipy:", scipy.__version__)

## 3. 데이터 불러오기

In [ ]:
from sklearn.preprocessing import LabelEncoder

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
feature_cols = [c for c in train.columns if c not in (ID_COL, TARGET)]
assert list(test.columns) == [ID_COL] + feature_cols   # 형식 확인만

X = train[feature_cols]
X_test = test[feature_cols]
le = LabelEncoder().fit(train[TARGET])                  # train 정답에만 fit
y = le.transform(train[TARGET])
print("train", X.shape, "| test", X_test.shape, "| classes", len(le.classes_))

## 4. 【② 피처】 전처리·피처 · 담당 ② 홍시윤
`src/preprocessing.py`, `src/features.py`의 최종 함수를 그대로 붙여 넣어요 (화요일 12시 확정본).

In [ ]:
# ===== 【② 피처】 시작 =====
# 출처: src/preprocessing.py, src/features.py (dev 기준) + 02_features "4-2. E" 피처 구성
#       핫스팟·위치 구간·정확한 변이 위치: notebooks/A_model/10_lgbm_g3_3model_equal_ensemble (H_data/03_hotspot_bins 재현)
import re
from collections import Counter
from scipy.sparse import csr_matrix, hstack
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer

WT = "WT"


def binarize_mutations(X: pd.DataFrame) -> pd.DataFrame:
    """[팀 기본 피처] WT 또는 결측 = 0, 변이 = 1."""
    return (X.notna() & (X != WT)).astype(np.int8)


def mutation_count(X: pd.DataFrame) -> pd.Series:
    """환자별 변이가 있는 유전자 개수."""
    return binarize_mutations(X).sum(axis=1).rename("n_mutated_genes")


def mutation_type_counts(X):
    """환자별 변이 유형 개수 (n_missense, n_synonymous, n_nonsense, n_frameshift, n_other)."""
    feature_names = ["n_missense", "n_synonymous", "n_nonsense", "n_frameshift", "n_other"]

    # 결측치와 WT를 제외한 실제 변이만 추출, 한 칸에 여러 변이면 각각 분리
    mutations = X.stack().dropna()
    mutations = mutations[mutations != "WT"]
    tokens = mutations.astype(str).str.split().explode().dropna()

    # 변이 표기 패턴에 따라 유형 분류
    def classify(token):
        if "fs" in token.lower():
            return "n_frameshift"
        if re.fullmatch(r"[A-Z]\d+\*", token):
            return "n_nonsense"
        if re.fullmatch(r"[A-Z]\d+[A-Z]", token):
            if token[0] == token[-1]:
                return "n_synonymous"
            return "n_missense"
        return "n_other"

    types = tokens.map(classify)
    counts = pd.crosstab(tokens.index.get_level_values(0), types)
    return counts.reindex(index=X.index, columns=feature_names, fill_value=0).astype("int32")


def amino_acid_pair_counts(X: pd.DataFrame) -> pd.DataFrame:
    """환자별 아미노산 치환 쌍 개수 (예: V600E -> aa_V_E), 400개 고정 열."""
    amino_acids = "ACDEFGHIKLMNPQRSTVWY"
    pairs = [f"aa_{a}_{b}" for a in amino_acids for b in amino_acids]

    mutations = X.stack().dropna()
    mutations = mutations[mutations != "WT"]
    tokens = mutations.astype(str).str.split().explode().dropna()

    # 아미노산 치환 표기 추출 (같은 아미노산끼리는 제외)
    matches = tokens.str.extract(r"^([A-Z])\d+([A-Z])$")
    valid = (
        matches[0].isin(list(amino_acids))
        & matches[1].isin(list(amino_acids))
        & (matches[0] != matches[1])
    )
    matches = matches.loc[valid]
    pair_names = "aa_" + matches[0] + "_" + matches[1]

    counts = pd.crosstab(pair_names.index.get_level_values(0), pair_names)
    return counts.reindex(index=X.index, columns=pairs, fill_value=0).astype("int32")


def gene_truncation_flags(X: pd.DataFrame) -> pd.DataFrame:
    """유전자별 단백질 절단형 변이(Nonsense, Frameshift) 여부 0/1."""
    def has_truncation(value):
        if pd.isna(value):
            return 0
        tokens = str(value).split()
        return int(any(
            "fs" in token.lower() or re.fullmatch(r"[A-Z]\d+\*", token)
            for token in tokens
        ))

    result = X.apply(lambda col: col.map(has_truncation))
    result.columns = [f"trunc_{col}" for col in X.columns]
    return result.astype("int8")


def make_E(X: pd.DataFrame) -> pd.DataFrame:
    """E 피처 9,174열: 0/1 + log1p(변이 수) + log1p(변이 유형 5개) + log1p(아미노산 쌍 400개) + 유전자별 끊김형."""
    parts = [binarize_mutations(X), np.log1p(mutation_count(X)),
             np.log1p(mutation_type_counts(X)), np.log1p(amino_acid_pair_counts(X)),
             gene_truncation_flags(X)]
    out = pd.concat(parts, axis=1).astype(np.float32)
    assert not out.columns.has_duplicates and out.index.equals(X.index)
    return out

def make_features_E_sparse(X):
    """E 피처 희소행렬."""
    return csr_matrix(make_E(X).to_numpy(dtype=np.float32))


# ----- 핫스팟 (유전자별 아미노산 위치 35개 + NPM1 frameshift 1개 = 36개) -----
# 출처: Vogelstein et al., Science 2013 / Chang et al., Nat Biotechnol 2016 (cancerhotspots.org)
#       IDH1·IDH2: Yan et al., NEJM 2009 / NPM1: Falini et al., NEJM 2005 / EGFR A289·G598: Lee et al., PLoS Med 2006
HOTSPOTS = {
    "IDH1": ["R132"],
    "IDH2": ["R140", "R172"],
    "BRAF": ["V600"],
    "HRAS": ["Q61", "G13", "G12"],
    "KIT": ["D816", "N822", "K642"],
    "CTNNB1": ["S33", "S37", "S45", "T41", "G34", "D32"],
    "U2AF1": ["S34"],
    "PIK3CA": ["E545", "H1047", "E542"],
    "EGFR": ["L858", "G598", "A289"],
    "RET": ["M918"],
    "RAC1": ["P29"],
    "SPOP": ["F133", "W131", "F102", "Y87"],
    "EZH2": ["Y641", "Y646"],
    "TP53": ["R175", "R248", "R273"],
    "PTEN": ["R130"],
}
assert sum(map(len, HOTSPOTS.values())) == 35


def make_hotspot_features(X):
    """환자별 핫스팟 변이 여부 0/1 (36열)."""
    hotspot_data = {}
    for gene, positions in HOTSPOTS.items():
        if gene not in X.columns:
            raise ValueError(f"유전자 컬럼 없음: {gene}")
        tokens = X[gene].fillna("WT").astype(str).str.split()
        for position in positions:
            hotspot_data[f"hotspot_{gene}_{position}"] = tokens.map(
                lambda variants: int(any(v.startswith(position) for v in variants))
            ).astype(np.int8)

    if "NPM1" not in X.columns:
        raise ValueError("유전자 컬럼 없음: NPM1")
    hotspot_data["hotspot_NPM1_fs"] = (
        X["NPM1"].fillna("WT").astype(str)
        .str.contains("fs", case=False, regex=False).astype(np.int8)
    )
    return pd.DataFrame(hotspot_data, index=X.index)


# ----- 위치 구간 (유전자별 아미노산 위치를 50 단위로 묶음, 학습 데이터 5명 이상만) -----
def extract_position(variant):
    match = re.search(r"\d+", variant)
    return int(match.group()) if match else None


def extract_position_bins(X, bin_size=50):
    rows, cols, gene_bins = [], [], set()
    for row_idx, (_, row) in enumerate(X.iterrows()):
        patient_bins = set()
        for gene, value in row.items():
            if gene == "ID":
                continue
            if pd.isna(value) or str(value) == "WT":
                continue
            for variant in str(value).split():
                pos = extract_position(variant)
                if pos is not None:
                    patient_bins.add(f"{gene}_bin_{pos // bin_size}")
        for feature in patient_bins:
            rows.append(row_idx)
            cols.append(feature)
            gene_bins.add(feature)
    return rows, cols, gene_bins


class PositionBinTransformer(BaseEstimator, TransformerMixin):
    """학습 데이터(fit)에서 환자 min_patients명 이상인 위치 구간만 선택."""
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        _, cols, _ = extract_position_bins(X, bin_size=self.bin_size)
        counts = Counter(cols)
        self.selected_bins_ = sorted(f for f, c in counts.items() if c >= self.min_patients)
        self.bin_to_idx_ = {f: i for i, f in enumerate(self.selected_bins_)}
        return self

    def transform(self, X):
        rows, cols, _ = extract_position_bins(X, bin_size=self.bin_size)
        sel_rows, sel_cols = [], []
        for row_idx, feature in zip(rows, cols):
            col_idx = self.bin_to_idx_.get(feature)
            if col_idx is not None:
                sel_rows.append(row_idx)
                sel_cols.append(col_idx)
        data = np.ones(len(sel_rows), dtype=np.float32)
        return csr_matrix((data, (sel_rows, sel_cols)),
                          shape=(len(X), len(self.selected_bins_)), dtype=np.float32)


class FeaturesEWithPositionBins(BaseEstimator, TransformerMixin):
    """LR 입력: E + 위치 구간."""
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size, min_patients=self.min_patients).fit(X)
        return self

    def transform(self, X):
        return hstack([make_features_E_sparse(X), self.position_transformer_.transform(X)],
                      format="csr", dtype=np.float32)


# ----- 정확한 변이 위치 (V600E -> BRAF_V600, 학습 데이터 5명 이상만) -----
EXACT_POSITION_RE = re.compile(r"^([A-Za-z])(\d+)")


def patient_exact_positions(X):
    result = []
    for row in X.itertuples(index=False, name=None):
        found = set()
        for gene, value in zip(X.columns, row):
            if gene in ("ID", "SUBCLASS") or pd.isna(value) or str(value).strip().upper() == "WT":
                continue
            for token in str(value).split():
                match = EXACT_POSITION_RE.match(token)
                if match:
                    found.add(f"{gene}_{match.group(1).upper()}{match.group(2)}")
        result.append(found)
    return result


class ExactPositionTransformer(BaseEstimator, TransformerMixin):
    """학습 데이터(fit)에서 환자 min_patients명 이상인 정확한 변이 위치만 선택."""
    def __init__(self, min_patients=5):
        self.min_patients = min_patients

    def fit(self, X, y=None):
        counts = Counter(key for keys in patient_exact_positions(X) for key in keys)
        self.selected_ = sorted(k for k, v in counts.items() if v >= self.min_patients)
        self.mapping_ = {key: i for i, key in enumerate(self.selected_)}
        return self

    def transform(self, X):
        rows, cols = [], []
        for i, keys in enumerate(patient_exact_positions(X)):
            for key in keys:
                j = self.mapping_.get(key)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        return csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)),
                          shape=(len(X), len(self.selected_)), dtype=np.float32)


class FeaturesEHotspot3BinExact(BaseEstimator, TransformerMixin):
    """LightGBM 입력: E + 핫스팟×3 + 위치 구간 + 정확한 변이 위치."""
    def __init__(self, bin_size=50, min_patients=5, hotspot_multiplier=3):
        self.bin_size = bin_size
        self.min_patients = min_patients
        self.hotspot_multiplier = hotspot_multiplier

    def fit(self, X, y=None):
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size, min_patients=self.min_patients).fit(X)
        self.exact_transformer_ = ExactPositionTransformer(min_patients=self.min_patients).fit(X)
        return self

    def transform(self, X):
        base = make_features_E_sparse(X)
        hotspot = csr_matrix(make_hotspot_features(X).to_numpy(dtype=np.float32) * self.hotspot_multiplier)
        bins = self.position_transformer_.transform(X)
        exact = self.exact_transformer_.transform(X)
        return hstack([base, hotspot, bins, exact], format="csr", dtype=np.float32)
# ===== 【② 피처】 끝 =====

## 5. 【③ 모델】 모델 정의 · 담당 ③ 안주형
최종 모델과 하이퍼파라미터를 숫자로 적어요. 현재: 제출 11 (가중치 XGBoost E + LR E·위치 구간 + LightGBM g3, LB 0.42103)

In [ ]:
# ===== 【③ 모델】 시작 =====
# 출처: notebooks/A_model/10_lgbm_g3_3model_equal_ensemble (제출 11, LB 0.42103)
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

N_CLASSES = 26
XGB_PARAMS = dict(n_estimators=250, max_depth=3, learning_rate=0.05, subsample=0.8,
                  colsample_bytree=0.8, objective="multi:softprob", num_class=N_CLASSES,
                  eval_metric="mlogloss", tree_method="hist", device="cpu",   # 재현용 CPU 고정
                  random_state=SEED, n_jobs=-1)
LR_PARAMS = dict(C=0.1, class_weight="balanced", max_iter=3000, solver="lbfgs")
LGBM_PARAMS = dict(n_estimators=800, learning_rate=0.02, num_leaves=15,
                   min_child_samples=5, colsample_bytree=0.2, subsample=0.8,
                   subsample_freq=1, class_weight="balanced", random_state=SEED,
                   verbosity=-1, n_jobs=-1)


class InvSqrtWeightedXGB(XGBClassifier):
    """클래스 가중치 1/√(클래스별 환자 수), 학습에 들어온 y로만 계산."""
    def fit(self, X, y, **kwargs):
        y = np.asarray(y, dtype=int)
        counts = np.bincount(y, minlength=N_CLASSES)
        w = 1.0 / np.sqrt(counts[y])
        return super().fit(X, y, sample_weight=(w / w.mean()).astype(np.float32), **kwargs)


def make_xgb():
    """가중치 XGBoost: E 피처."""
    return Pipeline([("features", FunctionTransformer(make_E)),
                     ("model", InvSqrtWeightedXGB(**XGB_PARAMS))])


def make_lr():
    """LR: E + 위치 구간."""
    return Pipeline([("features", FeaturesEWithPositionBins(bin_size=50, min_patients=5)),
                     ("model", LogisticRegression(**LR_PARAMS))])


def make_lgbm():
    """LightGBM g3: E + 핫스팟×3 + 위치 구간 + 정확한 변이 위치."""
    return Pipeline([("features", FeaturesEHotspot3BinExact(bin_size=50, min_patients=5, hotspot_multiplier=3)),
                     ("model", LGBMClassifier(**LGBM_PARAMS))])


MODELS = {"xgb_E_weighted": make_xgb, "lr_E_position": make_lr, "lgbm_g3": make_lgbm}
# ===== 【③ 모델】 끝 =====

## 6. (선택) 교차검증 · 팀 공통 get_folds (쌍둥이 그룹 StratifiedGroupKFold)

In [ ]:
from sklearn.base import clone
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold

# 팀 공통 검증 (src/validation.py): 쌍둥이(전체 변이 동일, 변이 2개 이상, KIPAN/KIRC 또는 GBMLGG/LGG)를 같은 fold로 묶음
TWIN_PAIRS = (("KIPAN", "KIRC"), ("GBMLGG", "LGG"))


def twin_groups(X, y, classes):
    names = np.asarray(classes)[np.asarray(y)].astype(str)
    raw = X.reset_index(drop=True)
    row_hash = pd.util.hash_pandas_object(raw, index=False).to_numpy()
    n_mut = (raw.notna() & raw.ne("WT") & raw.ne("")).sum(axis=1).to_numpy()
    groups = np.arange(len(raw), dtype=np.int64)
    for a, b in TWIN_PAIRS:
        eligible = np.flatnonzero((n_mut >= 2) & np.isin(names, [a, b]))
        buckets = {}
        for i in eligible:
            buckets.setdefault(int(row_hash[i]), []).append(int(i))
        for indices in buckets.values():
            if len(indices) < 2 or not ({a, b} <= set(names[indices])):
                continue
            for i in indices:
                if groups[i] != i:
                    continue
                same = [j for j in indices if raw.iloc[j].equals(raw.iloc[i])]
                if {a, b} <= set(names[same]):
                    groups[same] = min(same)
    return groups


def get_folds(y, X, classes):
    groups = twin_groups(X, y, classes)
    sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    folds = list(sgkf.split(X, y, groups=groups))
    for tr, va in folds:
        assert not (set(groups[tr]) & set(groups[va])), "같은 그룹이 학습·검증에 동시에 들어감"
    return folds


def run_cv(model, X, y, folds):
    scores = []
    for tr, va in folds:
        m = clone(model).fit(X.iloc[tr], y[tr])           # fold의 학습 부분에만 fit
        scores.append(f1_score(y[va], m.predict(X.iloc[va]), average="macro"))
    print(f"CV Macro F1 = {np.mean(scores):.4f} (std {np.std(scores):.4f})")
    return scores

if RUN_CV:
    folds = get_folds(y, X, le.classes_)
    for name, make in MODELS.items():
        print(name); run_cv(make(), X, y, folds)

## 7. 전체 train으로 학습 → test 예측 확률

In [ ]:
import time

try:
    from tqdm import tqdm   # 진행 막대 (글자 모드, 없으면 막대 없이 진행)
except ImportError:
    tqdm = None


def lgbm_progress(total):
    """LightGBM 나무 학습 진행 막대 (결과에는 영향 없음)."""
    bar = tqdm(total=total, desc="  LightGBM 나무", leave=True)
    def callback(env):
        bar.update(1)
        if env.iteration + 1 == env.end_iteration:
            bar.close()
    return callback


test_probas = {}
t_all = time.time()
for name, make in MODELS.items():
    t0 = time.time()
    print(name, "학습 시작")
    fit_kw = {}
    if name == "lgbm_g3" and tqdm is not None:
        fit_kw = {"model__callbacks": [lgbm_progress(LGBM_PARAMS["n_estimators"])]}
    model = make().fit(X, y, **fit_kw)                 # train 전체에만 fit
    test_probas[name] = model.predict_proba(X_test)
    print(name, f"학습 완료 ({(time.time() - t0) / 60:.1f}분)")
print(f"전체 학습·예측 {(time.time() - t_all) / 60:.1f}분")

## 8. 【④ 경량·앙상블】 최종 예측 결합 · 담당 ④ 박유정
모델이 하나면 그대로, 여러 개면 정한 가중치로 평균해요. 현재: 1/3씩 (제출 11 기준, notebooks/A_model/10)

In [ ]:
# ===== 【④ 경량·앙상블】 시작 =====
ENSEMBLE_WEIGHTS = {"xgb_E_weighted": 1/3, "lr_E_position": 1/3, "lgbm_g3": 1/3}   # 제출 11 (LB 0.42103), 같은 비중

names = list(test_probas)
w = (np.ones(len(names)) / len(names) if ENSEMBLE_WEIGHTS is None
     else np.array([ENSEMBLE_WEIGHTS[n] for n in names]) / sum(ENSEMBLE_WEIGHTS.values()))
final_proba = sum(wi * test_probas[n] for wi, n in zip(w, names))
# ===== 【④ 경량·앙상블】 끝 =====

## 9. 제출 파일 저장

In [ ]:
sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
assert len(sub) == len(test) and (sub[ID_COL].values == test[ID_COL].values).all()
sub[TARGET] = le.inverse_transform(final_proba.argmax(axis=1))
sub.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("저장:", OUTPUT_PATH, sub.shape)
sub.head()